In [ ]:
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt

RESULTS_V2  = "../results_v2/gsm8k"
RESULTS_V1  = "../results/gsm8k"   # homo_r8 only — eval rank unchanged for homo methods
METHODS  = ["homo_r8", "hetero_pad", "flexlora", "spa_m"]
SEEDS    = [42, 43, 44]
COLORS   = {"homo_r8": "#4CAF50", "hetero_pad": "#FF9800", "flexlora": "#2196F3", "spa_m": "#E91E63"}
LABELS   = {"homo_r8": "FedAvg r=8", "hetero_pad": "Hetero-Pad", "flexlora": "FlexLoRA", "spa_m": "SPA-M (Ours)"}

def load(method, seed):
    # homo_r8: V1 and V2 identical (all ranks=8, eval rank/weighting unchanged)
    dirs = [RESULTS_V1, RESULTS_V2] if method == "homo_r8" else [RESULTS_V2]
    for d in dirs:
        matches = glob.glob(os.path.join(d, f"{method}_seed{seed}*.json"))
        if matches:
            with open(matches[0]) as f:
                return json.load(f)
    return None

data = {}
for m in METHODS:
    for s in SEEDS:
        d = load(m, s)
        if d:
            data[(m, s)] = d

METRIC = 'exact_match' if 'exact_match' in next(iter(data.values()))['rounds'][0] else 'accuracy'
print(f"Loaded {len(data)} runs | metric: {METRIC}")
for (m, s) in sorted(data): print(f"  {m} seed={s}")

In [ ]:
# Summary table
summary = {m: {"mean_l5": [], "final": [], "best": []} for m in METHODS}
print(f"{'Method':<14} {'Seed':<6} {'Mean-L5★':>10} {'Final':>10} {'Best':>10}")
print("-" * 55)
for m in METHODS:
    for s in SEEDS:
        if (m, s) not in data: continue
        vals = [r[METRIC] for r in data[(m, s)]['rounds']]
        ml5, fin, best = np.mean(vals[-5:]), vals[-1], max(vals)
        summary[m]['mean_l5'].append(ml5)
        summary[m]['final'].append(fin)
        summary[m]['best'].append(best)
        print(f"{m:<14} {s:<6} {ml5*100:>9.2f}% {fin*100:>9.2f}% {best*100:>9.2f}%")
    print()

print("=" * 55)
print(f"{'Method':<14} {'n':<4} {'Mean-L5★':>13} {'Final':>13} {'Best':>13}")
print("-" * 60)
for m in METHODS:
    ml5s, fins, bsts = summary[m]['mean_l5'], summary[m]['final'], summary[m]['best']
    if not ml5s: continue
    print(f"{m:<14} {len(ml5s):<4} "
          f"{np.mean(ml5s)*100:>5.2f}±{np.std(ml5s)*100:.2f}%  "
          f"{np.mean(fins)*100:>5.2f}±{np.std(fins)*100:.2f}%  "
          f"{np.mean(bsts)*100:>5.2f}±{np.std(bsts)*100:.2f}%")

In [ ]:
# Convergence curves
fig, ax = plt.subplots(figsize=(9, 5))
for m in METHODS:
    runs = [[r[METRIC] for r in data[(m, s)]['rounds']] for s in SEEDS if (m, s) in data]
    if not runs: continue
    arr = np.array(runs)
    mu, std = arr.mean(0), arr.std(0)
    x = np.arange(1, len(mu) + 1)
    ax.plot(x, mu*100, label=f"{LABELS[m]} (n={len(arr)})", color=COLORS[m], linewidth=2)
    ax.fill_between(x, (mu-std)*100, (mu+std)*100, alpha=0.15, color=COLORS[m])

ax.set_xlabel("Round"); ax.set_ylabel("Exact Match (%)")
ax.set_title("GSM8K V2 — Math Reasoning | Qwen2.5-7B | α=0.5")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("gsm8k_v2_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Bar chart
methods_with_data = [m for m in METHODS if summary[m]['mean_l5']]
means = [np.mean(summary[m]['mean_l5'])*100 for m in methods_with_data]
stds  = [np.std(summary[m]['mean_l5'])*100  for m in methods_with_data]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([LABELS[m] for m in methods_with_data], means, yerr=stds,
              capsize=5, color=[COLORS[m] for m in methods_with_data],
              alpha=0.85, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel("Mean-L5 Exact Match (%)")
ax.set_title("GSM8K V2 — Mean-L5 ★")
ax.set_ylim(0, max(means)*1.2)
plt.xticks(rotation=15, ha='right'); plt.tight_layout()
plt.savefig("gsm8k_v2_bar.png", dpi=150, bbox_inches="tight")
plt.show()